# Detecção de Estados de Torneiras com YOLO — Versão Simples (1 split)  🚰

## Residência em Inteligência Artificial — Visão Computacional
### Desafio Semana 5 — Detecção de Estados com YOLO (versão direta: train/test único)

---

Este notebook treina um modelo **YOLO** para detectar o **estado** de uma torneira:

| Classe | id | Significado |
|--------|----|-------------|
| `torneira_aberta`  | 0 | Torneira **aberta** (em uso / fluxo de água) |
| `torneira_fechada` | 1 | Torneira **fechada** (desligada) |

O dataset foi criado pelo grupo (fotos próprias), anotado no **Label Studio**
(formato YOLO) e contém **85 imagens** com variações de iluminação, ângulo,
distância e fundo.

### 🎯 O que esta versão faz de diferente
É o ponto de partida **mais simples**: **remove as imagens duplicadas** e faz
**um único split treino/validação** (train/test split estratificado por classe).
Sem validação cruzada — mas com uma **seção (Seção 12) que explica os
experimentos com KFold**, por que os resultados variaram tanto e por que isso é
enganoso.

> Os outros dois notebooks aprofundam a avaliação:
> [`torneiras_estados_yolo.ipynb`](./torneiras_estados_yolo.ipynb) (5-fold por
> hash) e
> [`torneiras_estados_yolo_split_integrante.ipynb`](./torneiras_estados_yolo_split_integrante.ipynb)
> (5-fold por integrante).

### 🏭 Conexão com a indústria
Detectar se uma torneira/válvula está **aberta ou fechada** equivale a problemas
industriais reais: **monitoramento de válvulas/registros** em tubulações e plantas
(válvula aberta/fechada → segurança e controle de vazão), inspeção de atuadores em
linhas automatizadas, leitura de estado em painéis. Um erro de leitura de estado
(válvula que deveria estar fechada e está aberta) significa vazamento, desperdício
ou risco — por isso a detecção automática por visão computacional é tão útil.

---
> **Como usar:** `Ambiente de execução → Executar tudo` (use uma **GPU T4**).


## 1. Setup — instalação e checagem de ambiente

In [ ]:
# Instala a versão mais recente do Ultralytics (YOLO11)
%pip install -q "ultralytics>=8.3.0"

import ultralytics, torch
ultralytics.checks()
print("\nTorch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sem GPU — vá em 'Ambiente de execução → Alterar o tipo de ambiente → T4 GPU'.")


## 2. Configuração do experimento

Hiperparâmetros centralizados. Os defaults já estão calibrados para este dataset.

In [ ]:
from pathlib import Path

# --- Origem do dataset (repositório do grupo no GitHub) ---
REPO_URL    = "https://github.com/joaowinderfeldbussolotto/postgrad-cv-projects.git"
REPO_BRANCH = "claude/wonderful-fermi-mwei17"  # o dataset só existe nesta branch (não está no main)
REPO_DIR    = "postgrad-cv-projects"
DATA_SUBDIR = "project5/dataset"        # pasta com images/, labels/, classes.txt

# --- Modelo base (transfer learning a partir do COCO) ---
# n=leve | s=recomendado | m=mais pesado. Modelos maiores (m/l/x) foram testados
# neste dataset e overfitam (ver Seção 14) — por isso ficamos em yolo11s.
MODEL    = "yolo11s.pt"

# --- Hiperparâmetros de treino ---
IMGSZ    = 768        # 640 é mais rápido; 768 preserva o detalhe do registro/alavanca
EPOCHS   = 100
PATIENCE = 30         # early-stopping: para sozinho no melhor ponto
BATCH    = 16         # use -1 para batch automático conforme a VRAM
SEED     = 42
VAL_FRAC = 0.2        # fração do conjunto (já deduplicado) usada para validação

CLASS_NAMES = ["torneira_aberta", "torneira_fechada"]


## 3. Obter o dataset

Funciona em dois modos: clona o repositório (Colab) ou usa a pasta local (se já
estiver dentro do repo).

In [ ]:
import subprocess

def resolve_dataset_dir():
    for cand in [Path(DATA_SUBDIR), Path("..") / DATA_SUBDIR, Path("dataset")]:
        if (cand / "images").exists() and (cand / "labels").exists():
            return cand.resolve()
    cand = Path(REPO_DIR) / DATA_SUBDIR
    if (cand / "images").exists():
        return cand.resolve()
    print(f"Clonando o repositório do dataset (branch {REPO_BRANCH})...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL], check=True)
    return (Path(REPO_DIR) / DATA_SUBDIR).resolve()

DATASET_DIR = resolve_dataset_dir()
IMAGES_DIR  = DATASET_DIR / "images"
LABELS_DIR  = DATASET_DIR / "labels"

n_imgs   = len(list(IMAGES_DIR.glob("*.*")))
n_labels = len(list(LABELS_DIR.glob("*.txt")))
print("Dataset:", DATASET_DIR)
print(f"Imagens: {n_imgs} | Labels: {n_labels}")
assert n_imgs > 0, "Nenhuma imagem encontrada!"


## 4. Exploração do dataset — distribuição das classes

In [ ]:
from collections import Counter

def label_path_for(img_path):
    return LABELS_DIR / (img_path.stem + ".txt")

images = sorted([p for p in IMAGES_DIR.glob("*.*")
                 if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

img_main_class = {}
class_box_counter = Counter()
for img in images:
    lp = label_path_for(img)
    cls_ids = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if line.strip():
                c = int(float(line.split()[0])); cls_ids.append(c); class_box_counter[c] += 1
    img_main_class[img] = Counter(cls_ids).most_common(1)[0][0] if cls_ids else -1

print("Boxes por classe:")
for cid, name in enumerate(CLASS_NAMES):
    print(f"  {cid} {name}: {class_box_counter[cid]}")

import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.bar([CLASS_NAMES[c] for c in sorted(class_box_counter)],
        [class_box_counter[c] for c in sorted(class_box_counter)],
        color=["#2a9d8f", "#e76f51"])
plt.title("Distribuição de classes (nº de bounding boxes)")
plt.ylabel("quantidade"); plt.tight_layout(); plt.show()


In [ ]:
# Visualiza algumas imagens com suas bounding boxes
import cv2, random

def draw_boxes(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = label_path_for(img_path)
    colors = [(42,157,143), (231,111,81)]
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if not line.strip(): continue
            c, xc, yc, bw, bh = map(float, line.split()); c = int(c)
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), colors[c], 3)
            cv2.putText(img, CLASS_NAMES[c], (x1, max(20,y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, colors[c], 2)
    return img

random.seed(SEED)
sample = random.sample(images, min(6, len(images)))
plt.figure(figsize=(14, 8))
for i, img in enumerate(sample):
    plt.subplot(2, 3, i+1); plt.imshow(draw_boxes(img)); plt.axis("off")
    plt.title(img.name[:22], fontsize=8)
plt.tight_layout(); plt.show()


## 5. Detecção e **remoção** de imagens duplicadas

O export do Label Studio contém a **mesma foto exportada mais de uma vez** (cada
cópia anotada de forma levemente diferente). Se uma cópia cair no treino e a
outra (pixel-a-pixel idêntica) na validação, o modelo **vê a imagem de validação
durante o treino** — um vazamento que infla a métrica.

Aqui a solução é a mais simples possível: calculamos o **hash do conteúdo** de
cada imagem e **mantemos apenas uma cópia por hash**. Com o conjunto já
deduplicado, o split treino/validação fica trivial e **sem risco de leak** — não
precisamos de lógica *group-aware*.

In [ ]:
import hashlib
from collections import defaultdict

def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

hash_of = {img: file_hash(img) for img in images}

groups_by_hash = defaultdict(list)
for img in images:
    groups_by_hash[hash_of[img]].append(img)

# Mantém 1 representante por hash (o de nome menor, p/ ser determinístico)
unique_images = sorted(min(g, key=lambda p: p.name) for g in groups_by_hash.values())

n_dup_removed = len(images) - len(unique_images)
print(f"Imagens originais     : {len(images)}")
print(f"Imagens únicas (hash) : {len(unique_images)}")
print(f"Cópias duplicadas removidas: {n_dup_removed} "
      f"({n_dup_removed/len(images):.0%} do dataset original)")

# Distribuição de classes após a deduplicação
dedup_dist = Counter(img_main_class[i] for i in unique_images)
print("\nClasses após dedup:",
      {CLASS_NAMES[c]: dedup_dist[c] for c in sorted(dedup_dist)})


## 6. Split train/test e montagem do dataset YOLO

`build_yolo_dataset` copia imagens/labels para a estrutura esperada pelo YOLO e
escreve o `data.yaml`. O split é um **train/test split estratificado por classe**
(`sklearn.train_test_split`) sobre o conjunto **já deduplicado** — sem duplicatas,
um split simples já é suficiente e seguro.

In [ ]:
import shutil, yaml
from sklearn.model_selection import train_test_split

def build_yolo_dataset(train_imgs, val_imgs, out_dir):
    """Cria out_dir/{images,labels}/{train,val} + data.yaml e retorna o caminho do yaml."""
    out_dir = Path(out_dir).resolve()
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split in ["train", "val"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def populate(img_list, split):
        for img in img_list:
            shutil.copy(img, out_dir / "images" / split / img.name)
            lp = label_path_for(img)
            dst = out_dir / "labels" / split / (img.stem + ".txt")
            dst.write_text(lp.read_text() if lp.exists() else "")

    populate(train_imgs, "train")
    populate(val_imgs, "val")

    data_yaml = {"path": str(out_dir), "train": "images/train", "val": "images/val",
                 "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
    return yaml_path

# Split estratificado por classe sobre o conjunto deduplicado
y = [img_main_class[i] for i in unique_images]
train_imgs, val_imgs = train_test_split(
    unique_images, test_size=VAL_FRAC, stratify=y, random_state=SEED)

# Segurança: nenhuma imagem nos dois lados
assert not (set(train_imgs) & set(val_imgs)), "Imagem em treino E validação!"
assert not ({hash_of[i] for i in train_imgs} & {hash_of[i] for i in val_imgs}), \
    "Hash idêntico em treino E validação!"

DATA_YAML = build_yolo_dataset(train_imgs, val_imgs, "ds_torneiras")
print(f"Split → treino {len(train_imgs)} | val {len(val_imgs)}")
print("Treino por classe:", {CLASS_NAMES[c]: sum(img_main_class[i]==c for i in train_imgs)
                              for c in range(len(CLASS_NAMES))})
print("Val    por classe:", {CLASS_NAMES[c]: sum(img_main_class[i]==c for i in val_imgs)
                              for c in range(len(CLASS_NAMES))})


## 7. Sweep de augmentation (treinos curtos para escolher a melhor config)

### Augmentation calibrada para o problema

Torneira é "1 objeto centralizado, pista de estado sutil" (posição da
alavanca/água visível). Augmentations pensadas para COCO multi-objeto (mosaic,
mixup) **destroem essa pista** — por isso ficam desligadas em **todas** as
variações abaixo:

| Param | Motivo |
|-------|--------|
| `mosaic=0.0` | mosaic junta 4 imagens e encolhe a torneira → a pista de estado some |
| `mixup=0.0` | mixup mistura aberta+fechada → ruído de rótulo no que classificamos |
| `erasing=0.0` | não apagar a alavanca/registro (a pista visual) |

### Por que um sweep
Sem mais fotos para coletar, a **augmentation** é uma das poucas alavancas que
não depende de dados novos. Em vez de fixar à mão um único conjunto de
parâmetros, treinamos **4 variações curtas** (`SWEEP_EPOCHS` épocas, só para
comparar — não é o modelo final) sobre o mesmo split da Seção 6 e escolhemos a
que vence no `mAP50` (empate decidido pelo `mAP50-95`) para o treino completo
da Seção 8.

In [ ]:
from ultralytics import YOLO
import pandas as pd

SWEEP_EPOCHS   = 30
SWEEP_PATIENCE = 15

AUG_CANDIDATES = {
    "baseline": dict(
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        degrees=8.0, translate=0.1, scale=0.3, fliplr=0.5, flipud=0.0,
        mosaic=0.0, mixup=0.0, erasing=0.0,
    ),
    "mais_geometria": dict(   # mais tolerância a ângulo/distância
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        degrees=15.0, translate=0.15, scale=0.4, fliplr=0.5, flipud=0.0,
        mosaic=0.0, mixup=0.0, erasing=0.0,
    ),
    "mais_cor": dict(         # mais robustez a iluminação
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
        degrees=8.0, translate=0.1, scale=0.3, fliplr=0.5, flipud=0.0,
        mosaic=0.0, mixup=0.0, erasing=0.0,
    ),
    "leve": dict(             # controle: augmentation ~40% mais fraca
        hsv_h=0.01, hsv_s=0.3, hsv_v=0.25,
        degrees=5.0, translate=0.06, scale=0.18, fliplr=0.5, flipud=0.0,
        mosaic=0.0, mixup=0.0, erasing=0.0,
    ),
}

sweep_rows = []
for nome, cfg in AUG_CANDIDATES.items():
    print(f"\n=== Sweep: {nome} ===")
    m = YOLO(MODEL)
    m.train(data=str(DATA_YAML), epochs=SWEEP_EPOCHS, patience=SWEEP_PATIENCE,
             imgsz=IMGSZ, batch=BATCH, seed=SEED, optimizer="auto", cos_lr=True,
             cache=True, project="runs_torneiras", name=f"sweep_{nome}",
             exist_ok=True, verbose=False, plots=False, **cfg)
    r = m.val(data=str(DATA_YAML), split="val", verbose=False)
    sweep_rows.append({"config": nome, "mAP50": r.box.map50, "mAP50-95": r.box.map,
                        "P": r.box.mp, "R": r.box.mr})

sweep_df = pd.DataFrame(sweep_rows).sort_values(["mAP50", "mAP50-95"], ascending=False)
print("\n================ RESULTADO DO SWEEP ================")
print(sweep_df.to_string(index=False))

melhor = sweep_df.iloc[0]["config"]
AUG = AUG_CANDIDATES[melhor]
print(f"\n✅ Configuração escolhida para o treino final: '{melhor}'")


## 8. Treino final do modelo

Treino completo (`EPOCHS`/`PATIENCE`, early-stopping + decaimento cosseno do
LR) usando a configuração de augmentation **escolhida pelo sweep** da Seção 7.

In [ ]:
model = YOLO(MODEL)
model.train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
            batch=BATCH, seed=SEED, optimizer="auto", cos_lr=True, cache=True,
            project="runs_torneiras", name="simples", exist_ok=True,
            verbose=False, plots=True, **AUG)

SAVE_DIR     = Path(model.trainer.save_dir)
BEST_WEIGHTS = SAVE_DIR / "weights" / "best.pt"
print("\nPesos salvos em:", BEST_WEIGHTS)


## 9. Métricas do modelo final

`mAP@50`, `mAP@50-95`, **P** (precision) e **R** (recall) no conjunto de
validação — **sem** e **com Test-Time Augmentation (TTA)**. TTA roda a imagem em
múltiplas escalas/espelhamentos e combina as predições: é um ganho potencial
"de graça", sem retreinar, mas só vale a pena se realmente melhorar o `mAP50`
*neste* dataset — por isso comparamos antes de decidir se ela entra na
inferência (Seções 11/13).

In [ ]:
res     = model.val(data=str(DATA_YAML), split="val", verbose=False)
res_tta = model.val(data=str(DATA_YAML), split="val", augment=True, verbose=False)

print("================ MÉTRICAS (validação) ================")
print(f"{'':14s} {'sem TTA':>10s} {'com TTA':>10s}")
print(f"{'mAP50':14s} {res.box.map50:10.3f} {res_tta.box.map50:10.3f}")
print(f"{'mAP50-95':14s} {res.box.map:10.3f} {res_tta.box.map:10.3f}")
print(f"{'P (precision)':14s} {res.box.mp:10.3f} {res_tta.box.mp:10.3f}")
print(f"{'R (recall)':14s} {res.box.mr:10.3f} {res_tta.box.mr:10.3f}")

USE_TTA = bool(res_tta.box.map50 >= res.box.map50)
print(f"\n{'✅ TTA ajuda' if USE_TTA else '⚠️ TTA não ajuda'} neste dataset "
      f"(mAP50 com TTA {'>=' if USE_TTA else '<'} sem TTA) → "
      f"usando augment={USE_TTA} nas Seções 11 e 13.")


## 10. Diagnóstico — matriz de confusão e curvas

**Como ler a matriz de confusão:**
- Muita confusão **entre `torneira_aberta` e `torneira_fechada`** → problema de
  **estado**: o modelo acha a torneira mas erra se está aberta/fechada (pista sutil →
  precisa de fotos mais nítidas do registro / mais dados).
- Muita confusão com **`background`** (linha/coluna extra) → problema de
  **localização/detecção**: o modelo não encontra a torneira (ângulo/iluminação ruins).

In [ ]:
from IPython.display import Image, display

print("Resultados do treino:", SAVE_DIR)
for plot in ["confusion_matrix.png", "confusion_matrix_normalized.png",
             "BoxPR_curve.png", "BoxF1_curve.png", "results.png"]:
    p = SAVE_DIR / plot
    if p.exists():
        print(plot); display(Image(filename=str(p), width=560))


## 11. Inferência no conjunto de validação

Comparação visual: **anotação real** (esquerda) × **predição do modelo** (direita).

> **Regra de classe única:** uma torneira não pode estar aberta **e** fechada ao
> mesmo tempo. Como há só 1 torneira por imagem neste projeto, usamos
> `agnostic_nms=True` (funde caixas sobrepostas de classes diferentes) e
> `max_det=1` (mantém só a detecção de **maior confiança**) — assim a predição
> mostrada é sempre **um único estado decidido**, nunca as duas classes juntas.
>
> **Test-Time Augmentation:** `augment=USE_TTA` (decidido na Seção 9) — só ativa
> o TTA aqui se ele comprovadamente melhorou o `mAP50` na validação.

In [ ]:
best = YOLO(str(BEST_WEIGHTS))
val_image_files = sorted((Path(DATA_YAML).parent / "images" / "val").glob("*.*"))
sample_val = val_image_files[:min(5, len(val_image_files))]

plt.figure(figsize=(12, 4*len(sample_val)))
for i, img in enumerate(sample_val):
    pred = best.predict(str(img), imgsz=IMGSZ, conf=0.25,
                         agnostic_nms=True, max_det=1, augment=USE_TTA, verbose=False)[0]
    pred_img = cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB)
    plt.subplot(len(sample_val), 2, 2*i+1)
    plt.imshow(draw_boxes(img)); plt.axis("off"); plt.title(f"Real — {img.name[:20]}", fontsize=9)
    plt.subplot(len(sample_val), 2, 2*i+2)
    plt.imshow(pred_img); plt.axis("off"); plt.title("Predição do modelo", fontsize=9)
plt.tight_layout(); plt.show()


## 12. Sobre os testes com validação cruzada (KFold) — e por que são enganosos

Antes de chegar a este split único, rodamos **validação cruzada 5-fold** para ter
uma métrica "mais robusta". O resultado foi este:

```
 fold  mAP50  mAP50-95     P     R
    0  0.982     0.594 0.952 0.893
    1  0.417     0.169 0.392 0.723
    2  0.988     0.657 0.893 0.938
    3  0.442     0.187 0.440 0.450
    4  0.683     0.386 0.502 0.850

================ RESULTADO 5-FOLD (média ± desvio) ================
mAP50    : 0.702 ± 0.278
mAP50-95 : 0.399 ± 0.225
P (precision) : 0.636 ± 0.265
R (recall)    : 0.771 ± 0.196
```

### Por que isso aconteceu
O desvio é enorme (mAP50 de **0.42 a 0.99**) e quase **bimodal**: alguns folds vão
a ~0.98, outros despencam para ~0.42. A causa **não é o modelo** — é a composição
do dataset:

- As 85 imagens vêm de apenas **10 integrantes** (`s01`…`s10`), e **3 deles
  concentram ~62%** (s04=22, s05=17, s06=14). Cada integrante ≈ *uma* torneira,
  cozinha e iluminação → **poucas cenas independentes**.
- Mesmo sem nenhuma imagem idêntica cruzando treino/val, fotos **diferentes da
  mesma torneira** caem dos dois lados. O modelo aprende a reconhecer *aquela
  cena* e acerta fácil → folds ~0.98. Quando o fold valida uma cena pouco
  representada no treino → ~0.42.

### Por que a métrica é enganosa
- A **média sozinha (0.702) esconde** que a distribuição é bimodal: nenhum fold
  ficou perto de 0.70 — eles ficaram nos extremos.
- Um **único split "de sorte"** poderia reportar ~0.98 e parecer excelente, sem
  generalizar de verdade. O mAP aqui mede tanto *"que cena caiu na validação"*
  quanto a real qualidade do modelo.
- Por isso este notebook usa **um split estratificado simples + dedup** para
  treinar/demonstrar, e **não** trata a CV como um número final confiável.

### Como poderíamos evoluir
1. **Mais integrantes/torneiras independentes** (alvo 200–300 imagens *únicas*) —
   a maior alavanca; é o que estabiliza a métrica.
2. Validar com **group-split por integrante** (cada pessoa só de um lado), para
   medir generalização honesta a uma torneira nova → ver
   [`torneiras_estados_yolo_split_integrante.ipynb`](./torneiras_estados_yolo_split_integrante.ipynb).
3. Ler sempre a métrica pela **média ± desvio**, nunca por um fold isolado.

## 13. Teste em imagens NOVAS  📸

Item 4.4 do desafio: tire **3 a 5 fotos novas** (abertas e fechadas) e faça upload
para testar a generalização em imagens que o modelo **nunca viu**.

> Mesma **regra de classe única** da Seção 11: `agnostic_nms=True, max_det=1`
> garante **1 torneira → 1 estado**, resolvendo a favor da detecção de **maior
> confiança** em caso de duas detecções conflitantes (aberta + fechada). Também
> usa `augment=USE_TTA` (TTA), decidido na Seção 9.

In [ ]:
CONF = 0.25   # confiança mínima da detecção (ajuste se houver muitos FP/FN)
IOU  = 0.50

new_images = []
try:
    from google.colab import files
    print("Selecione de 3 a 5 fotos novas de torneiras...")
    uploaded = files.upload()
    new_images = [Path(name) for name in uploaded.keys()]
except Exception:
    folder = Path("novas_imagens")
    if folder.exists():
        new_images = sorted([p for p in folder.glob("*.*")
                             if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    print(f"{len(new_images)} imagem(ns) encontradas em 'novas_imagens/'.")


In [ ]:
if new_images:
    plt.figure(figsize=(7, 6*len(new_images)))
    for i, img in enumerate(new_images):
        res = best.predict(str(img), imgsz=IMGSZ, conf=CONF, iou=IOU,
                            agnostic_nms=True, max_det=1, augment=USE_TTA, verbose=False)[0]
        out = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        dets = [f"{CLASS_NAMES[int(b.cls)]} ({float(b.conf):.2f})" for b in res.boxes]
        plt.subplot(len(new_images), 1, i+1)
        plt.imshow(out); plt.axis("off")
        plt.title(f"{img.name}  →  " + (", ".join(dets) if dets else "nada detectado"),
                  fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print("Nenhuma imagem nova carregada — rode a célula anterior para fazer upload.")


## 14. Conclusões e análise

### Sobre a confiabilidade da métrica
O dataset tinha **duplicatas exatas** (fotos exportadas 2x do Label Studio). Aqui
elas são **removidas** (Seção 5) e o modelo treina num **split estratificado
único** sobre o conjunto deduplicado — sem leak de imagens idênticas. A Seção 12
explica por que a validação cruzada deu números **bimodais e enganosos** (poucas
cenas independentes) e por que a leitura honesta exige mais dados ou um split por
integrante.

### Melhorias aplicadas sem fotos novas
Sem mais imagens para coletar, as duas alavancas usadas aqui foram o **sweep de
augmentation** (Seção 7 — testa variações curtas e escolhe a que generaliza
melhor no split atual) e o **Test-Time Augmentation** (Seção 9 — só ativado na
inferência quando comprovadamente melhora o `mAP50`). Ambas extraem mais
desempenho do mesmo dataset, mas não substituem o ganho real que viria de mais
imagens/torneiras independentes (Seção 12).

### E rodar um YOLO XL / yolo26 xlarge?
**Não ajudaria** — pelo contrário. Em poucas dezenas de imagens, mais parâmetros =
**mais overfitting** e treino mais lento, sem ganho de generalização. O gargalo é
**quantidade/qualidade de dados e configuração**, não a capacidade do modelo.
`yolo11s` é a escolha certa.

### Sobre o pré-processamento do Roboflow
- ❌ **"Isolate Objects"**: é técnica de *classificação* (recorta cada objeto numa
  imagem). Para **detecção** quebra o modelo — remove o contexto e faz a box virar a
  imagem inteira; em fotos reais (cena completa) a detecção falha. **Não usar.**
- ❌ **"Grayscale 100%"**: descarta cor (água/metal) e o backbone COCO é RGB. **Não
  usar** como pré-processamento fixo. *(Grayscale como augmentation de 15% é ok.)*
- ⚠️ **"Stretch 640"** distorce proporção → preferir o *letterbox* padrão do YOLO.
- ⚠️ **"90° rotate"** é irreal para torneira (não fica de lado) → remover. Brilho/flip/
  rotação ±15° leves são bons.

### Como melhorar ainda mais (maior alavanca primeiro)
1. **Mais dados** (alvo 200–300 imgs *únicas*, sem duplicatas) com variação de
   ângulo/luz/fundo — maior ganho.
2. **Anotação consistente** + evitar imagens ambíguas (torneira aberta sem água
   visível parece fechada).
3. **Fotos que mostrem bem o registro/alavanca** (a pista de estado).
4. Ajustar `CONF`/`IOU` na inferência conforme o erro observado.

### 🏭 Paralelo industrial (resumo)
O mesmo pipeline — detectar **aberto/fechado** de um objeto — aplica-se ao
**monitoramento de válvulas e registros industriais**, onde identificar
automaticamente o estado de um atuador previne vazamentos, desperdício e falhas.
